In [2]:
import csv
from dataclasses import Field


def find_code(csv_file_path, district_name) -> str:
    """
    根据区域名字返回编码
    :param csv_file_path: 区域信息文件
    :param district_name: 区域名称
    :return:
    """
    district_map = {}
    with open(csv_file_path, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            district_code = row['districtcode'].strip()
            district = row['district'].strip()
            if district not in district_map:
                district_map[district] = district_code
    return district_map.get(district_name)

'440100'

In [9]:
from langchain_core.callbacks import CallbackManagerForToolRun
from typing import Type, Any, Optional
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field


# 定义参数
class WeatherInputArgs(BaseModel):
    location: str = Field(..., description='用于查询天气的位置信息')


# 定义工具
class WeatherTool(BaseTool):
    name: str = "weather_tool"
    description: str = "用于查询天气的工具"
    args_schema: Type[WeatherInputArgs] = WeatherInputArgs

    def _run(self, location: str, run_manager: Optional[CallbackManagerForToolRun] = None, ) -> Any:
        district_code = find_code('data/weather_district_id.csv', location)
        # 调用天气查询接口
        return "{'district_code': district_code, 'location': location, 'weather': '晴'}"


In [10]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
import os
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

agent_executor = create_agent(model, [WeatherTool()])
agent_executor.invoke({
    "messages":[HumanMessage(content="今天北京天气怎么样?")]
})


{'messages': [HumanMessage(content='今天北京天气怎么样?', additional_kwargs={}, response_metadata={}, id='3f56f46e-e2bd-4ff8-9b36-ccd5e9d46ce0'),
  AIMessage(content='我来帮您查询今天北京的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 321, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 321}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V3.2', 'system_fingerprint': '', 'id': '019d5663c5fd8aa49581261c922e26d5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5663-c503-7581-a617-be440ffe14de-0', tool_calls=[{'name': 'weather_tool', 'args': {'location': '北京'}, 'id': '019d5663d1e66bf0b338e3ce462dbf18', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m